# 08 DoS versus DDoS: windowed aggregation

**Purpose.** About 97% of the multiclass error is one confusion between DoS and DDoS. This notebook tests whether temporal aggregation over neighbouring flows recovers the distinction, and which features carry it.

**Data layout.** Each CICIoMT2024 CSV is a concatenation of capture blocks, one per label, in capture order. A window statistic uses the previous 50 and 200 flows of the same capture only (causal).
**Protocols.** `chrono`: first 85% of every capture trains, last 15% tests. `heldout`: whole captures are held out (13 of 50, three repeats), the stricter test.

**Findings stored in `results/`.** On the 25 selected per-flow features the DoS/DDoS accuracy is 0.78 (chrono) / 0.59 (held-out); with windows it is 0.96 / 0.82. All 45 released features reach 99.9%, because inter-arrival time (IAT), a feature the selection discarded, is nearly constant with a class-specific offset of about 0.03%, which looks like a recording artefact; it is not used as a detection feature. Removing IAT leaves the windowed result unchanged.
The test uses a Random Forest, not the hybrid model, and each capture window contains a single attack class.

**To rerun** set `CICIOMT_DIR` to the folder with `CIC_IoMT_2024_WiFi_MQTT_train.csv` and `..._test.csv` (not redistributed; see the dataset page).

In [1]:
# --- setup: work from the laptop/ folder so that relative paths resolve ---
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
pd.set_option("display.precision", 4)
print("working folder:", ROOT)

working folder: <repo>\local_analysis


In [2]:
CICIOMT_DIR = None     # e.g. r"D:\data\CIC IoMT 2024 WiFi MQTT"
import subprocess
if CICIOMT_DIR:
    for feats in ["all", "kept25", "noIAT"]:
        subprocess.run([sys.executable, "dos_ddos_window_test.py", CICIOMT_DIR], env={**os.environ, "FEATURES": feats})
    subprocess.run([sys.executable, "dos_ddos_feature_diagnosis.py", CICIOMT_DIR])

In [3]:
rows = []
for name in ["all", "kept25", "noIAT"]:
    r = json.load(open(f"results/dos_ddos_window_test_{name}.json"))
    rows.append({"features": {"all": "all 45", "kept25": "25 selected", "noIAT": "44 (all but IAT)"}[name],
                 "chrono per-flow": r["chrono"]["per_flow"]["acc"], "chrono + windows": r["chrono"]["per_flow+window"]["acc"],
                 "held-out per-flow": r["heldout_mean_sd_per_flow"][0], "held-out + windows": r["heldout_mean_sd_per_flow+window"][0],
                 "held-out SD (windows)": r["heldout_mean_sd_per_flow+window"][1]})
pd.DataFrame(rows)

,features,chrono per-flow,chrono + windows,held-out per-flow,held-out + windows,held-out SD (windows)
0,all 45,0.9997,0.9999,0.9995,0.9996,0.0004
1,25 selected,0.7761,0.9582,0.5866,0.8235,0.1459
2,44 (all but IAT),0.7768,0.9590,0.5815,0.8275,0.1427


In [4]:
D = json.load(open("results/dos_ddos_feature_diagnosis.json"))
print("dropped features:", D["dropped_features"])
print({k: round(D[k]["mean"], 4) for k in ["all_45", "kept_25", "dropped_20"]})
pd.Series(D["single_feature_depth3_acc"]).head(8).to_frame("single-feature depth-3 tree accuracy")

dropped features: ['Protocol Type', 'Srate', 'Drate', 'ece_flag_number', 'cwr_flag_number', 'Telnet', 'SMTP', 'SSH', 'IRC', 'DHCP', 'IGMP', 'IPv', 'LLC', 'Tot size', 'IAT', 'Number', 'Magnitue', 'Radius', 'Covariance', 'Weight']
{'all_45': 0.9994, 'kept_25': 0.5819, 'dropped_20': 0.9994}


,single-feature depth-3 tree accuracy
IAT,0.9998
Header_Length,0.6748
Std,0.6628
Radius,0.6624
Covariance,0.6490
UDP,0.6421
Rate,0.6277
Srate,0.6277
